# Fake News Detection - Standalone Environment
Run the cells below sequentially. The first few cells will automatically generate the project structure inside your Colab environment.

In [ ]:
!mkdir -p src
%%writefile src/utils.py
import yaml
import torch

def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

def get_device(device_str):
    if device_str == "auto":
        if torch.cuda.is_available():
            return torch.device('cuda')
        elif torch.backends.mps.is_available():
            return torch.device('mps')
        else:
            return torch.device('cpu')
    return torch.device(device_str)


In [ ]:
!mkdir -p src
%%writefile src/data.py
import os
import os.path as osp
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected

def get_loaders(config):
    """
    Creates train, val, and test data loaders based on the provided configuration.
    """
    data_cfg = config['data']
    path = osp.join(osp.dirname(osp.realpath(__file__)), '..', data_cfg['data_dir'])
    
    # Common transform to ensure graphs are undirected
    transform = ToUndirected()

    # Use the standard UPFD dataset, which will automatically download if missing
    train_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'train', transform)
    val_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'val', transform)
    test_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'test', transform)

    train_loader = DataLoader(train_dataset, batch_size=data_cfg['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=data_cfg['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=data_cfg['batch_size'], shuffle=False)

    return train_loader, val_loader, test_loader, train_dataset


In [ ]:
!mkdir -p src
%%writefile src/trainer.py
import torch
import torch.nn.functional as F
from tqdm import tqdm

class Trainer:
    def __init__(self, model, optimizer, device, train_loader, val_loader, test_loader):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

    def _get_model_kwargs(self, data):
        kwargs = {
            'x': data.x,
            'edge_index': data.edge_index,
            'batch': data.batch
        }
        if hasattr(data, 'sentiment'):
            kwargs['sentiment_features'] = data.sentiment
        return kwargs

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        for data in self.train_loader:
            data = data.to(self.device)
            self.optimizer.zero_grad()
            out = self.model(**self._get_model_kwargs(data))
            loss = F.nll_loss(out, data.y)
            loss.backward()
            self.optimizer.step()
            total_loss += float(loss) * data.num_graphs
        return total_loss / len(self.train_loader.dataset)

    @torch.no_grad()
    def test(self, loader):
        self.model.eval()
        total_correct = total_examples = 0
        for data in loader:
            data = data.to(self.device)
            out = self.model(**self._get_model_kwargs(data))
            pred = out.argmax(dim=-1)
            total_correct += int((pred == data.y).sum())
            total_examples += data.num_graphs
        return total_correct / total_examples

    def fit(self, epochs):
        for epoch in range(1, epochs + 1):
            loss = self.train_epoch()
            train_acc = self.test(self.train_loader)
            val_acc = self.test(self.val_loader)
            test_acc = self.test(self.test_loader)
            print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
                  f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')



In [ ]:
!mkdir -p src/models
%%writefile src/models/cmcg.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCoAttentionLayer(nn.Module):
    def __init__(self, hidden_channels1, hidden_channels2, num_heads=2, dropout_rate=0.01):
        super(MultiHeadCoAttentionLayer, self).__init__()
        self.num_heads = num_heads
        self.hidden_channels1 = hidden_channels1
        self.hidden_channels2 = hidden_channels2

        self.query1 = nn.Linear(hidden_channels1, hidden_channels1 * num_heads, bias=False)
        self.key1 = nn.Linear(hidden_channels1, hidden_channels1 * num_heads, bias=False)
        self.value1 = nn.Linear(hidden_channels1, hidden_channels1 * num_heads, bias=False)

        self.query2 = nn.Linear(hidden_channels2, hidden_channels2 * num_heads, bias=False)
        self.key2 = nn.Linear(hidden_channels2, hidden_channels2 * num_heads, bias=False)
        self.value2 = nn.Linear(hidden_channels2, hidden_channels2 * num_heads, bias=False)

        self.out1 = nn.Linear(hidden_channels1 * num_heads, hidden_channels1)
        self.out2 = nn.Linear(hidden_channels2 * num_heads, hidden_channels2)
        
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x1, x2):
        Q1 = self.query1(x1).view(-1, self.num_heads, self.hidden_channels1)
        K2 = self.key2(x2).view(-1, self.num_heads, self.hidden_channels2)
        V2 = self.value2(x2).view(-1, self.num_heads, self.hidden_channels2)

        Q2 = self.query2(x2).view(-1, self.num_heads, self.hidden_channels2)
        K1 = self.key1(x1).view(-1, self.num_heads, self.hidden_channels1)
        V1 = self.value1(x1).view(-1, self.num_heads, self.hidden_channels1)

        attn_scores1 = torch.matmul(Q1, K2.transpose(-2, -1)) / (self.hidden_channels2 ** 0.5)
        attn_weights1 = self.dropout(F.softmax(attn_scores1, dim=-1))
        attended_x1 = torch.matmul(attn_weights1, V2)

        attn_scores2 = torch.matmul(Q2, K1.transpose(-2, -1)) / (self.hidden_channels1 ** 0.5)
        attn_weights2 = self.dropout(F.softmax(attn_scores2, dim=-1))
        attended_x2 = torch.matmul(attn_weights2, V1)

        attended_x1 = attended_x1.view(-1, self.num_heads * self.hidden_channels1)
        attended_x2 = attended_x2.view(-1, self.num_heads * self.hidden_channels2)

        out_x1 = self.out1(attended_x1) + x1
        out_x2 = self.out2(attended_x2) + x2

        return out_x1, out_x2


In [ ]:
!mkdir -p src/models
%%writefile src/models/classifier.py
import torch.nn as nn
from torch.nn import Linear

class MLPClassifier(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # A simple MLP head. This can be made more complex if needed.
        self.lin = Linear(in_channels, out_channels)

    def forward(self, x):
        h = self.lin(x)
        return h.log_softmax(dim=-1)


In [ ]:
!mkdir -p src/models
%%writefile src/models/text_encoders.py
import torch
from torch.nn import Linear

class TextEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.lin = Linear(in_channels, hidden_channels)

    def forward(self, x, batch):
        # Get the root node (news content) features of each graph:
        # In UPFD, the first node of each graph in the batch is the root node.
        # However, to be robust across batches, we find the first occurrence of each batch index.
        
        # This logic identifies the indices of the first node for each graph in the batch
        root_indices = (batch[1:] - batch[:-1]).nonzero(as_tuple=False).view(-1)
        root_indices = torch.cat([root_indices.new_zeros(1), root_indices + 1], dim=0)
        
        news_features = x[root_indices]
        return self.lin(news_features).relu()


In [ ]:
!mkdir -p src/models
%%writefile src/models/gnn_encoders.py
import torch
from torch_geometric.nn import GATConv, GCNConv, SAGEConv, global_max_pool

class GNNEncoder(torch.nn.Module):
    def __init__(self, model_type, in_channels, hidden_channels):
        super().__init__()
        
        if model_type == 'GCN':
            self.conv = GCNConv(in_channels, hidden_channels)
        elif model_type == 'SAGE':
            self.conv = SAGEConv(in_channels, hidden_channels)
        elif model_type == 'GAT':
            self.conv = GATConv(in_channels, hidden_channels)
        else:
            raise ValueError(f"Unsupported GNN model type: {model_type}")

    def forward(self, x, edge_index, batch):
        h = self.conv(x, edge_index).relu()
        h = global_max_pool(h, batch)
        return h


In [ ]:
!mkdir -p src/models
%%writefile src/models/upfd_model.py
import torch
import torch.nn as nn
from src.models.gnn_encoders import GNNEncoder
from src.models.text_encoders import TextEncoder
from src.models.classifier import MLPClassifier
from src.models.cmcg import MultiHeadCoAttentionLayer

class SentimentEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # assuming sentiment feature is provided separately (e.g., shape [batch_size, in_channels])
        self.lin = nn.Linear(in_channels, hidden_channels)

    def forward(self, sentiment_features):
        return self.lin(sentiment_features).relu()

class UPFDModel(nn.Module):
    def __init__(self, config, in_channels, out_channels):
        super().__init__()
        model_cfg = config['model']
        self.use_gnn = model_cfg.get('use_gnn', True)
        self.use_text = model_cfg.get('use_text', True)
        self.use_sentiment = model_cfg.get('use_sentiment', False)
        self.use_cmcg = model_cfg.get('use_cmcg', False)
        
        hidden_channels = model_cfg['hidden_channels']

        self.gnn_encoder = None
        self.text_encoder = None
        self.sentiment_encoder = None
        combined_channels = 0

        if self.use_gnn:
            self.gnn_encoder = GNNEncoder(
                model_cfg['gnn_type'], in_channels, hidden_channels
            )
            combined_channels += hidden_channels

        if self.use_text:
            self.text_encoder = TextEncoder(in_channels, hidden_channels)
            combined_channels += hidden_channels

        if self.use_cmcg and self.use_gnn and self.use_text:
            self.co_attention = MultiHeadCoAttentionLayer(hidden_channels, hidden_channels, num_heads=2)
            # Channels remain the same after co-attention since we still concat them

        if self.use_sentiment:
            sentiment_dim = model_cfg.get('sentiment_dim', 1) # default to 1D sentiment score
            self.sentiment_encoder = SentimentEncoder(sentiment_dim, hidden_channels)
            combined_channels += hidden_channels

        if combined_channels == 0:
            raise ValueError("At least one encoder (GNN, Text, or Sentiment) must be enabled in the config.")

        self.classifier = MLPClassifier(combined_channels, out_channels)

    def forward(self, x, edge_index, batch, sentiment_features=None):
        embeddings = []
        h_gnn = None
        h_text = None

        if self.use_gnn:
            h_gnn = self.gnn_encoder(x, edge_index, batch)
            
        if self.use_text:
            h_text = self.text_encoder(x, batch)

        if self.use_cmcg and h_gnn is not None and h_text is not None:
            h_gnn, h_text = self.co_attention(h_gnn, h_text)
            
        if h_gnn is not None:
            embeddings.append(h_gnn)
        if h_text is not None:
            embeddings.append(h_text)

        if self.use_sentiment:
            if sentiment_features is None:
                # Fallback to zeros if not provided in the batch
                sentiment_features = torch.zeros(batch.max().item() + 1, self.sentiment_encoder.lin.in_features).to(x.device)
            h_sent = self.sentiment_encoder(sentiment_features)
            embeddings.append(h_sent)

        # Concatenate embeddings from all active encoders
        if len(embeddings) > 1:
            h = torch.cat(embeddings, dim=-1)
        else:
            h = embeddings[0]

        return self.classifier(h)


In [ ]:
%%writefile main.py
import torch
from src.utils import load_config, get_device
from src.data import get_loaders
from src.models.upfd_model import UPFDModel
from src.trainer import Trainer

def run_experiment(config):
    # Get device
    device = get_device(config['training']['device'])
    print(f"Using device: {device}")

    # Prepare data
    train_loader, val_loader, test_loader, dataset = get_loaders(config)
    
    # Initialize model
    model = UPFDModel(
        config=config,
        in_channels=dataset.num_features,
        out_channels=dataset.num_classes
    )
    
    # Optimizer
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=config['training']['lr'], 
        weight_decay=config['training']['weight_decay']
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        device=device,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader
    )
    
    # Start training
    trainer.fit(config['training']['epochs'])
    test_acc = trainer.test(trainer.test_loader)
    return test_acc

def main():
    # Load configuration
    config = load_config('config.yaml')
    run_experiment(config)

if __name__ == "__main__":
    main()


In [ ]:
%%writefile config.yaml
# UPFD Project Configuration

data:
  dataset: "politifact" # choices: ['politifact', 'gossipcop']
  feature: "spacy"      # choices: ['profile', 'spacy', 'bert', 'content']
  batch_size: 128
  data_dir: "dataset"

model:
  gnn_type: "GCN"      # choices: ['GCN', 'GAT', 'SAGE']
  hidden_channels: 128
  use_gnn: true
  use_text: true
  # Future extensions
  use_sentiment: false

training:
  lr: 0.001
  weight_decay: 0.01
  epochs: 60
  device: "auto" # 'auto', 'cuda', or 'cpu'


In [ ]:
%%writefile colab_experiments.py
import itertools
import pandas as pd
import time
import torch
import random
import numpy as np

# Ensure this script is run in an environment where main.py can be imported
from main import run_experiment

# Colab-specific imports for Google Sheets
try:
    from google.colab import auth
    from google.auth import default
    import gspread
    COLAB_ENV = True
except ImportError:
    print("Not running in Google Colab, or gspread/auth not installed. Sheet logging will be skipped/simulated.")
    COLAB_ENV = False

def setup_google_sheet(sheet_name="UPFD_Experiments"):
    if not COLAB_ENV:
        return None
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    
    try:
        sh = gc.open(sheet_name)
        worksheet = sh.sheet1
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(sheet_name)
        worksheet = sh.sheet1
        # Set up headers
        headers = ["Seed", "Dataset", "Feature", "Use GNN", "GNN Type", "Use Text", "Use CMCG (Co-Attention)", "Use Sentiment", "Epochs", "Accuracy"]
        worksheet.append_row(headers)
    
    return worksheet

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def main():
    worksheet = setup_google_sheet("Fake_News_Detection_Results")
    
    # Define the test matrix
    seeds = [42, 2026]
    datasets = ["gossipcop"] # Can add "politifact"
    features = ["bert"] # or "spacy"
    
    # Combinations of model architectures
    # Each tuple: (use_gnn, gnn_type, use_text, use_cmcg, use_sentiment)
    model_configs = [
        # Baseline Text only
        (False, "GCN", True, False, False),
        
        # Baseline GNN only (GCN & SAGE)
        (True, "GCN", False, False, False),
        (True, "SAGE", False, False, False),
        
        # GNN + Text (Early fusion / Concatenation)
        (True, "GCN", True, False, False),
        (True, "SAGE", True, False, False),
        
        # CMCG (Co-Attention between GNN and Text)
        (True, "SAGE", True, True, False),
        
        # Text + Sentiment
        (False, "GCN", True, False, True),
        
        # Full Model: GNN + Text + CMCG + Sentiment
        (True, "SAGE", True, True, True),
    ]

    for seed in seeds:
        for dataset in datasets:
            for feature in features:
                for use_gnn, gnn_type, use_text, use_cmcg, use_sentiment in model_configs:
                    set_seed(seed)
                    
                    config = {
                        "data": {
                            "dataset": dataset,
                            "feature": feature,
                            "batch_size": 128,
                            "data_dir": "dataset"
                        },
                        "model": {
                            "gnn_type": gnn_type,
                            "hidden_channels": 128,
                            "use_gnn": use_gnn,
                            "use_text": use_text,
                            "use_cmcg": use_cmcg,
                            "use_sentiment": use_sentiment,
                            "sentiment_dim": 1 # Example dummy sentiment dimension
                        },
                        "training": {
                            "lr": 0.001,
                            "weight_decay": 0.01,
                            "epochs": 30, # Reduced for testing, adjust as needed
                            "device": "auto"
                        }
                    }
                    
                    print(f"\n--- Running Experiment ---")
                    print(f"Seed: {seed}, Dataset: {dataset}, Feature: {feature}")
                    print(f"GNN: {use_gnn} ({gnn_type}), Text: {use_text}, CMCG: {use_cmcg}, Sentiment: {use_sentiment}")
                    
                    try:
                        acc = run_experiment(config)
                        print(f"Achieved Accuracy: {acc:.4f}")
                        
                        # Log to Google Sheets
                        if worksheet is not None:
                            row = [
                                seed, dataset, feature, 
                                use_gnn, gnn_type, use_text, use_cmcg, use_sentiment, 
                                config['training']['epochs'], float(acc)
                            ]
                            worksheet.append_row(row)
                            # Sleep briefly to avoid Google Sheets API rate limits
                            time.sleep(1)
                    except Exception as e:
                        print(f"Experiment failed: {e}")
                        if worksheet is not None:
                            row = [
                                seed, dataset, feature, 
                                use_gnn, gnn_type, use_text, use_cmcg, use_sentiment, 
                                config['training']['epochs'], f"ERROR: {str(e)}"
                            ]
                            worksheet.append_row(row)

if __name__ == "__main__":
    main()


# 1. Install Dependencies

In [ ]:
!pip install torch_geometric PyYAML gspread

# 2. Download and Extract Dataset
Fixes the 404 error from PyTorch Geometric.

In [ ]:
!mkdir -p dataset/gossipcop/raw
!curl -L -o dataset/gossipcop/raw/data.zip "https://data.pyg.org/datasets/upfd_gossipcop.zip"
!cd dataset/gossipcop/raw && unzip -o data.zip && rm data.zip

# 3. Run Experiments
This will authenticate with Google Sheets and start the training loops.

In [ ]:
!python colab_experiments.py